# Domain 3: Claude Code Configuration & Workflows
## Task Statement 3.4: Determine When to Use Plan Mode vs Direct Execution
**Certification Weight**: 18% Exam Weight | **Standard**: Claude 4.5 & Extended Thinking

## 🎯 Exam Objective
Understand when to invoke **Plan Mode** versus **Direct Execution**, how to leverage the **Explore Subagent** to isolate verbose codebase investigations, and apply iterative prompt refinement patterns.

---

## 🏗️ Architecture: Plan Mode vs Direct Execution Decision Tree

```mermaid
flowchart TD
    TaskStart["Incoming Development Task"] --> ScopeCheck{"Evaluate Scope & Ambiguity"}
    
    ScopeCheck -->|"Single-file fix<br>Known stack trace<br>Unambiguous bug"| DirectExec["Direct Execution Mode<br>(Fast surgical edits via Edit / Bash)"]
    
    ScopeCheck -->|"Multi-file refactor<br>Architectural trade-offs<br>> 5 files affected"| PlanMode["Plan Mode (/plan)<br>(Read-only exploration & user-approved plan)"]
    
    PlanMode --> LaunchExplore["Spawn Explore Subagent<br>(context: fork)"]
    LaunchExplore --> DeepSearch["Search AST, Grep call hierarchy,<br>examine dependencies"]
    DeepSearch --> CompactPlan["Synthesize Clean Architecture Plan<br>(Discard raw file dumps)"]
    
    CompactPlan --> UserApproval{"User Approves Plan?"}
    UserApproval -->|"Revisions requested"| RefinePlan["Refine Plan with Interview Technique"]
    RefinePlan --> UserApproval
    UserApproval -->|"Approved"| ExecutePhased["Execute Phased Changes in Codebase"]
```

---

## ⚠️ Critical Exam Distinctions & Anti-Patterns

| Strategy | When to Select on the Exam | Exam Trap to Reject |
|---|---|---|
| **Plan Mode (`/plan`)** | Large refactoring, cross-cutting architectural changes, or when multiple valid implementations exist. | **TRAP**: Diving straight into writing code on a 40-file refactor without creating an architectural plan first. |
| **Direct Execution** | Local bug fixes, well-isolated single-function modifications with deterministic test cases. | **TRAP**: Entering Plan Mode for trivial syntax fixes or 1-line typo corrections (needless friction and delay). |
| **Explore Subagent** | Large codebase discovery and dependency mapping. Runs with `context: fork` to protect main context. | **TRAP**: Dumping thousands of lines of raw grep output directly into the main conversation history. |
| **Iterative Refinement** | Pair ambiguous requirements with **2–3 concrete Input/Output examples** and the **Interview Pattern**. | **TRAP**: Typing repetitive generic follow-ups ("try again", "make it better") without providing test cases or constraints. |

---

## 💡 Architectural Decision Matrix

| Dimension | Direct Execution | Plan Mode (`/plan`) |
|---|---|---|
| **Typical File Scope** | 1–2 files | 3+ files or unknown dependency tree |
| **Tool Restrictions** | All allowed tools active | Read-only exploration until plan approval |
| **Context Strategy** | Inline modifications | Explore subagent with `context: fork` |
| **Reversibility** | High (isolated edit) | Low (broad structural changes) |

---

---

### 🧠 Claude 4.5 Extended Thinking & Parameter Standards (October 2026)
- **Model Standard**: Use `claude-sonnet-4-5` or `claude-haiku-4-5`.
- **Temperature Omission**: When `thinking` is enabled (`thinking={"type": "enabled", "budget_tokens": ...}`), **do not provide `temperature`**, `top_p`, or `top_k`. The Anthropic API rejects custom sampling values with an HTTP `400 Bad Request` error.
- **Thinking Continuity**: In multi-turn loops, assistant reasoning blocks (`type: "thinking"`) must be returned **unmodified with their cryptographic signature** in subsequent turns. Stripping thinking blocks causes an API `400 Bad Request` error.
- **Case Facts Preservation**: Immutable transactional data (Order IDs, customer accounts, amounts) must be maintained in a dedicated Case Facts block outside conversation summaries.

In [ ]:
# Step 1: Environment & Setup
import sys
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Dict, List

repo_root = Path.cwd()
while repo_root != repo_root.parent and not (repo_root / "pyproject.toml").exists():
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

print(" Environment initialized for TS 3.4 Plan Mode vs Direct Execution.")

### ⚖️ Step 2: Mode Evaluation Engine & Interview Prompts
We implement the decision logic evaluating whether a task warrants Plan Mode or Direct Execution.

In [ ]:
# Step 2: Mode Evaluation Logic
@dataclass
class WorkflowStrategyDecision:
    mode: str  # 'plan_mode' | 'direct_execution'
    rationale: str
    recommended_tools: List[str]


def evaluate_execution_mode(
    file_count_affected: int,
    has_architectural_tradeoffs: bool,
    has_known_stack_trace: bool,
    is_single_function_fix: bool,
) -> WorkflowStrategyDecision:
    """Determines whether to trigger Plan Mode vs Direct Execution."""
    if has_architectural_tradeoffs or file_count_affected > 3 or (not has_known_stack_trace and not is_single_function_fix):
        return WorkflowStrategyDecision(
            mode="plan_mode",
            rationale="Complex task involving multi-file impact or architectural trade-offs requires Plan Mode.",
            recommended_tools=["Read", "Grep", "Glob", "Explore"],
        )
    return WorkflowStrategyDecision(
        mode="direct_execution",
        rationale="Isolated single-file fix with clear stack trace is suitable for Direct Execution.",
        recommended_tools=["Read", "Edit", "Bash"],
    )

def build_interview_pattern_prompt(task_description: str) -> str:
    """Generates prompt instructing Claude to interview user before planning."""
    return (
        f"You are in Plan Mode for task: '{task_description}'. "
        "Do NOT write code or modify files yet. "
        "First, interview me by asking 2-3 clarifying questions with concrete trade-off options."
    )

print(" Mode evaluation engine initialized.")

### 🧪 Step 3: Interactive Simulation
1. Complex refactor -> Plan Mode.
2. Isolated bugfix -> Direct Execution.

In [ ]:
# Step 3: Simulation
# Scenario 1: Multi-file caching refactor
dec_plan = evaluate_execution_mode(
    file_count_affected=8,
    has_architectural_tradeoffs=True,
    has_known_stack_trace=False,
    is_single_function_fix=False,
)
print("Scenario 1 Mode:", dec_plan.mode)
print("Scenario 1 Rationale:", dec_plan.rationale)
print("Allowed Tools in Plan Mode:", dec_plan.recommended_tools)

# Scenario 2: Isolated ZeroDivisionError bugfix
dec_direct = evaluate_execution_mode(
    file_count_affected=1,
    has_architectural_tradeoffs=False,
    has_known_stack_trace=True,
    is_single_function_fix=True,
)
print("\nScenario 2 Mode:", dec_direct.mode)
print("Scenario 2 Rationale:", dec_direct.rationale)

### 🔬 Step 4: Verification Assertions
We assert that:
1. Multi-file refactor strictly routes to Plan Mode with read-only tools.
2. Single-file fix routes to Direct Execution.
3. Interview prompt instructs Claude to interview the user.

In [ ]:
# Step 4: Verification Checks
assert dec_plan.mode == "plan_mode"
assert "Explore" in dec_plan.recommended_tools or "Read" in dec_plan.recommended_tools
assert "Edit" not in dec_plan.recommended_tools, "Plan mode must use read-only exploration!"

assert dec_direct.mode == "direct_execution"
assert "Edit" in dec_direct.recommended_tools

interview_prompt = build_interview_pattern_prompt("Cache layer")
assert "interview me" in interview_prompt

print(" All verification assertions passed! Mode evaluation verified.")

### 💡 CCAR-F Exam Golden Rules
- **Rule 1**: Use **Plan Mode** for tasks touching >3 files, architectural redesigns, or ambiguous requirements.
- **Rule 2**: In Plan Mode, use **read-only tools** (`Read`, `Grep`, `Glob`) to explore; do not edit files without plan approval.
- **Rule 3**: Use **Direct Execution** only for single-file, low blast-radius bug fixes with clear stack traces.